# Reconstruction Example Using lftomo

Environment Requirements:

See x

In [1]:
from multiprocessing import shared_memory
from multiprocessing.managers import SharedMemoryManager
import os
from pathlib import Path
import sys

# Image Display
import cv2
from ipywidgets import Image
# Used for everything
import numpy as np
# Used to get processor data for setting multiprocessing.
import psutil
# Default file format is tif
import tifffile as tf

sys.path.insert(0, os.path.abspath('..'))

# Flexible logging to screen and file, in a parseable fixed-width format.
from lftomo.util.log import log
# Windowing to select only parts of an image at a time.
from lftomo.util.window import DataWindowSet, ProjOrder, ReconOrder, SinoOrder, get_tilt_pad, CTFov
# Handling flatfield backgrounds.
from lftomo.prep.flat import FLAT
# Shared Memory Handling
from lftomo.util.mem import SharedNP


## Setup

### Reconstruction Parameters

Starting with basics (input path, window size, output location)

In [2]:
log.header()
# smm = SharedMemoryManager()
# smm.start()

TYPE  |STEP                |   TIMESTAMP   |MEM USAGE|MEM FREE | STATEMENT 


In [3]:

window_size = psutil.cpu_count()
input_path = Path("..", "sample_data", "Sam19")
flat_path = Path("..", "sample_data", "Sam19_Flats")
output_path = Path("..", "sample_data", "Sam19_R")

fov = CTFov(Theta=None, Degrees=180, Count=1)

internal_dtype = np.float32		# Data type to use internally. 

In [4]:
data_set = DataWindowSet.from_folder(input_path, ProjOrder, window_size, fov)

### Flat Field Generation

In [5]:
flat_mem = SharedNP(f'flats_ipynb', internal_dtype, ProjOrder(len(FLAT), data_set.data_shape.Y, data_set.data_shape.X)).create()

with flat_mem as flat_set:
	for flat in list(FLAT):
		flat_set[flat.index, :, :] = tf.imread(flat_path.joinpath(f"{flat}_median.tiff")).astype(internal_dtype)
	
	# Precalculation of pregain and postgain with darks.
	flat_set[FLAT.PREGAIN.index] -= flat_set[FLAT.PREDARK.index]
	flat_set[FLAT.POSTGAIN.index] -= flat_set[FLAT.POSTDARK.index]


INFO  |Shared Memory       |00:00:00.337866|0002.00GB|0042.00GB|"Shared Memory Name flats_ipynb Pre-Existed"
INFO  |Shared Memory       |00:00:00.339256|0002.00GB|0042.00GB|"Shared Memory Name flats_ipynb Recreated"


In [7]:
if False:
	with flat_mem as flat_set:
		# Encode into in-memory PNG for Jupyter
		_, pregain = cv2.imencode('.png', flat_set[FLAT.PREGAIN.index])
		_, postgain = cv2.imencode('.png', flat_set[FLAT.POSTGAIN.index])

		tf.imwrite("pregain.tiff", flat_set[FLAT.PREGAIN.index])
		tf.imwrite("postgain.tiff", flat_set[FLAT.POSTGAIN.index])

	# This 'works' but requires leveling
	Image(value=pregain, width=data_set.data_shape.X, height=data_set.data_shape.Y)

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x08\x00\x00\x00\x08\x00\x08\x00\x00\x00\x00\x97\xcc\…

### Angle Calculation

In [ ]:
# No angle dropping so skipping for now.

## Center Finding

Steps:

1. Read Slice(s)
2. Data Cleanup
3. Center Calculation
4. Angle Calculation
5. Reconstruct Slices
6. Normalize Slices


In [ ]:
for CF_Interval in data_set.at_interval(8):
	# Read Slices
	log.write("Center Finding", f"{CF_Interval}")
	pass

TIME  |Center Finding      |00:00:00.895397|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 0"
TIME  |Center Finding      |00:00:00.898206|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 256"
TIME  |Center Finding      |00:00:00.899629|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 512"
TIME  |Center Finding      |00:00:00.902461|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 768"
TIME  |Center Finding      |00:00:00.904491|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 1024"
TIME  |Center Finding      |00:00:00.905948|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 1280"
TIME  |Center Finding      |00:00:00.907767|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 1536"
TIME  |Center Finding      |00:00:00.909881|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=1, X=2048) (uint16) at 1792"


## Reconstruction

Loop, looking at small windows each time.

1. Read Data
2. Clean Data (Pre-Rotation Steps)
3. Rotate Data
4. Clean Data (Post-Rotation Steps)
5. Reconstruct Slices
6. Normalize Slices

In [ ]:
for data in data_set:
	log.write("Reconstruction", f'{data}')

TIME  |Reconstruction      |00:00:01.022544|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 0"
TIME  |Reconstruction      |00:00:01.024858|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 12"
TIME  |Reconstruction      |00:00:01.026151|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 24"
TIME  |Reconstruction      |00:00:01.027838|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 36"
TIME  |Reconstruction      |00:00:01.029746|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 48"
TIME  |Reconstruction      |00:00:01.031053|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 60"
TIME  |Reconstruction      |00:00:01.032127|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 72"
TIME  |Reconstruction      |00:00:01.033074|0002.00GB|0042.00GB|"ProjOrder(Theta=1501, Y=12, X=2048) (uint16) at 84"
TIME  |Reconstruction      |00:00:01.034445|0002.00GB|0042.00GB|"

In [ ]:

flat_mem.close()
flat_mem.unlink()

STATUS|Shared Memory       |00:00:01.531233|0002.00GB|0042.00GB|"Shared Memory flats_ipynb Unlinked"


: 